# PixelCNN od podstaw: autoregresyjne generowanie obrazów CIFAR-10

> **Cel notebooka:** zbudować punkt odniesienia do nauki PixelCNN w PyTorch: od faktoryzacji rozkładu i masek przyczynowych, przez bloki rezydualne i trening, po analizę wyników, próbkowanie oraz dyskretną mieszaninę logistyczną.

PixelCNN modeluje prawdopodobieństwo obrazu jako iloczyn warunkowych prawdopodobieństw pikseli. To model **jawny**: przypisuje wynikową wartość logarytmu wiarygodności obserwowanym obrazom. Ceną za prostą funkcję celu jest sekwencyjne, powolne generowanie.

Notebook korzysta wyłącznie z PyTorch, torchvision, NumPy i Matplotlib. Dane CIFAR-10 są pobierane przez torchvision przy pierwszym uruchomieniu. Domyślnie kod używa GPU CUDA, jeśli jest dostępne; w przeciwnym razie działa na CPU. Trening pełnego modelu może być długi — parametry konfiguracyjne pozwalają zacząć od krótkiego przebiegu.

**Mapa rozdziałów**
1. Założenia, dane i wizualizacja
2. Autoregresyjna faktoryzacja i przyczynowość
3. Maskowane warstwy konwolucyjne i test braku przecieku
4. Bloki rezydualne oraz architektura PixelCNN
5. Dyskretna mieszanina logistyczna (DML) dla RGB
6. Trening, generowanie i analiza
7. Ograniczenia, eksperymenty i dalsze kroki


## 1. Przygotowanie środowiska

W razie braku bibliotek odkomentuj instalację w osobnej komórce. W typowym środowisku PyTorch wystarczy zainstalować wersję `torch` i `torchvision` odpowiednią dla platformy z [oficjalnej strony PyTorch](https://pytorch.org/get-started/locally/).


In [ ]:
import math
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. CIFAR-10: obrazy i reprezentacja

CIFAR-10 zawiera kolorowe obrazy 32×32 z dziesięciu klas. Etykiety nie będą wejściem do modelu: uczymy **bezwarunkowy** rozkład obrazów. Obrazy pozostają w skali 8-bitowej, ale wejście zamieniamy na liczby zmiennoprzecinkowe w `[0, 1]`. Nie standaryzujemy kanałów średnią i odchyleniem, ponieważ później modelujeć będziemy dyskretną intensywność każdego kanału.

Zbiór walidacyjny to wydzielona część oficjalnego zbioru treningowego. Zostawiamy test CIFAR-10 do końcowej oceny, jeśli chcemy porównywać warianty bez strojenia na teście.


In [ ]:
DATA_ROOT = Path("data")
BATCH_SIZE = 64
NUM_WORKERS = 0  # bezpieczny wybór dla notebooków na Windows
LIMIT_TRAIN = None  # np. 5000, aby szybko sprawdzić przepływ pracy
LIMIT_VAL = 2000

transform = transforms.ToTensor()  # uint8 -> float w [0, 1]
train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# Stały podział walidacyjny; generator zapewnia powtarzalność.
gen = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_full), generator=gen).tolist()
val_indices = perm[:LIMIT_VAL]
train_indices = perm[LIMIT_VAL:]
if LIMIT_TRAIN is not None:
    train_indices = train_indices[:LIMIT_TRAIN]
train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(train_full, val_indices)

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

class_names = train_full.classes
print(f"train={len(train_dataset):,}, val={len(val_dataset):,}, test={len(test_dataset):,}")


In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, images[:10], labels[:10]):
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 3. Autoregresyjna faktoryzacja i kolejność pikseli

Dla obrazu złożonego z `H×W` pikseli przyjmujemy porządek rasterowy: od lewej do prawej, od góry do dołu. Wówczas

`p(x) = ∏ᵢ p(xᵢ | x₁, …, xᵢ₋₁)`.

Sieć konwolucyjna może obliczyć parametry rozkładów dla wszystkich pozycji równolegle podczas treningu. Maski gwarantują, że predykcja w pozycji `i` nie korzysta z przyszłych pikseli. Podczas generowania nie znamy przyszłych wartości, więc musimy wypełniać obraz po kolei.

W RGB modelujemy rozkład **wektora koloru piksela** warunkowany wcześniejszymi pikselami. Wewnątrz tego wektora mieszanina logistyczna wprowadzi zależność kanałów: `G` zależy od `R`, a `B` od `R` i `G`.


## 4. Maskowane warstwy konwolucyjne

Maska typu **A** zeruje środek filtra wraz z całym obszarem przyszłym. Używamy jej w pierwszej warstwie, której wejściem są rzeczywiste piksele. Maska typu **B** dopuszcza środek, ale nadal blokuje przyszłe pozycje; stosujemy ją na ukrytych reprezentacjach. Dzięki temu informacja z bieżącego piksela nie trafia do własnej predykcji.

Nasza maska uwzględnia także konwolucję wielowarstwową (dylatacja i rozmiar jądra), a nie tylko typowe jądro 3×3.


In [ ]:
class MaskedConv2d(nn.Conv2d):
    """Konwolucja przyczynowa dla porządku rasterowego (typ A lub B)."""
    def __init__(self, *args, mask_type="B", **kwargs):
        if mask_type not in {"A", "B"}:
            raise ValueError("mask_type musi być 'A' albo 'B'")
        super().__init__(*args, **kwargs)
        kh, kw = self.kernel_size
        if kh % 2 == 0 or kw % 2 == 0:
            raise ValueError("Dla tej implementacji jądra muszą mieć nieparzysty rozmiar")
        mask = torch.ones_like(self.weight)
        cy, cx = kh // 2, kw // 2
        mask[:, :, cy + 1 :, :] = 0
        mask[:, :, cy, cx + 1 :] = 0
        if mask_type == "A":
            mask[:, :, cy, cx] = 0
        self.register_buffer("mask", mask)

    def forward(self, x):
        return F.conv2d(
            x, self.weight * self.mask, self.bias,
            self.stride, self.padding, self.dilation, self.groups
        )


In [ ]:
# Test przyczynowości: zmiana piksela (y, x) nie może zmienić wyjść
# w tej pozycji ani w pozycjach wcześniejszych w porządku rasterowym.
layer = MaskedConv2d(3, 4, kernel_size=3, padding=1, mask_type="A")
x1 = torch.randn(1, 3, 5, 5)
x2 = x1.clone()
x2[:, :, 2, 2] += 100
with torch.no_grad():
    y1, y2 = layer(x1), layer(x2)
# Piksele przed (2,2): wszystkie wiersze <2 oraz kolumny <2 w wierszu 2.
assert torch.allclose(y1[:, :, :2, :], y2[:, :, :2, :])
assert torch.allclose(y1[:, :, 2, :2], y2[:, :, 2, :2])
assert torch.allclose(y1[:, :, 2, 2], y2[:, :, 2, 2])  # maska A blokuje środek
print("Maska A blokuje bieżący i przyszłe piksele.")


## 5. Blok rezydualny i pole recepcyjne

Bloki rezydualne ułatwiają optymalizację głębszych sieci: uczą poprawki do reprezentacji wejściowej, zamiast odtwarzać całą reprezentację od zera. Każda konwolucja w ścieżce bloku ma maskę B. Normalizacja i aktywacja punktowa nie mieszają pozycji, więc nie naruszają przyczynowości.

W tej wersji stosujemy `1×1 → 3×3 → 1×1`. Filtr 3×3 powiększa pole recepcyjne o jeden piksel w każdym kierunku na blok. W praktycznych modelach używa się także większej liczby warstw, dylatacji lub osobnych pionowych i poziomych stosów, aby szybko objąć większy kontekst.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, hidden_channels=None):
        super().__init__()
        hidden_channels = hidden_channels or channels
        self.net = nn.Sequential(
            nn.ReLU(inplace=True),
            MaskedConv2d(channels, hidden_channels, kernel_size=1, mask_type="B"),
            nn.ReLU(inplace=True),
            MaskedConv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1, mask_type="B"),
            nn.ReLU(inplace=True),
            MaskedConv2d(hidden_channels, channels, kernel_size=1, mask_type="B"),
        )

    def forward(self, x):
        return x + self.net(x)


## 6. Dyskretna mieszanina logistyczna dla RGB

Kanały obrazu są dyskretne: `0, …, 255`. Zamiast przewidywać 256 klas niezależnie dla każdego kanału, używamy **mieszaniny logistycznych rozkładów dyskretnych**. Dla piksela model emituje `K` wag mieszaniny, średnie i skale dla RGB oraz współczynniki zależności kolorów. Składnik logistyczny jest ciągły, ale jego gęstość całkujemy po przedziale odpowiadającym każdemu poziomowi 8-bitowemu.

Parametry dla kanałów G i B są korygowane przez wcześniej próbkowane kanały:

- `μG ← μG + aRG·R`
- `μB ← μB + aRB·R + aGB·G`

Log-wiarygodności RGB w obrębie piksela sumują się warunkowo, następnie składniki mieszaniny łączymy przez `logsumexp`. Minimalizujemy ujemną log-wiarygodność w **nat**, a bity na wymiar (bits/dim) raportujemy jako `NLL / (H·W·C·ln 2)`.


In [ ]:
def discretized_mix_logistic_loss(params, x, num_mixtures):
    """Ujemna log-wiarygodność; params: (B, K*10, H, W), x w [-1, 1]."""
    b, _, h, w = params.shape
    params = params.permute(0, 2, 3, 1).contiguous()
    logit_probs = params[..., :num_mixtures]
    values = params[..., num_mixtures:].view(b, h, w, 3, 3 * num_mixtures)
    means = values[..., :num_mixtures]
    log_scales = torch.clamp(values[..., num_mixtures:2*num_mixtures], min=-7.0)
    coeffs = torch.tanh(values[..., 2*num_mixtures:])

    x = x.permute(0, 2, 3, 1).contiguous()  # wejście już w [-1, 1]
    xr, xg, xb = x.unbind(dim=-1)
    mr, mg, mb = means.unbind(dim=-2)
    sr, sg, sb = log_scales.unbind(dim=-2)
    crg, crb, cgb = coeffs.unbind(dim=-2)

    xr = xr.unsqueeze(-1)
    xg = xg.unsqueeze(-1)
    xb = xb.unsqueeze(-1)
    m_r = mr
    m_g = mg + crg * xr
    m_b = mb + crb * xr + cgb * xg

    inv_scale_r, inv_scale_g, inv_scale_b = torch.exp(-sr), torch.exp(-sg), torch.exp(-sb)
    half_bin = 1.0 / 255.0

    def channel_log_prob(xc, mean, log_scale, inv_scale):
        centered = xc - mean
        plus = inv_scale * (centered + half_bin)
        minus = inv_scale * (centered - half_bin)
        cdf_plus = torch.sigmoid(plus)
        cdf_minus = torch.sigmoid(minus)
        log_cdf_plus = F.logsigmoid(plus)
        log_one_minus_cdf_minus = F.logsigmoid(-minus)
        cdf_delta = cdf_plus - cdf_minus
        mid = inv_scale * centered
        log_pdf_mid = mid - log_scale - 2.0 * F.softplus(mid)
        log_cdf_delta = torch.where(
            cdf_delta > 1e-5,
            torch.log(torch.clamp(cdf_delta, min=1e-12)),
            log_pdf_mid + math.log(2.0 * half_bin),
        )
        log_prob = torch.where(xc < -0.999, log_cdf_plus,
                               torch.where(xc > 0.999, log_one_minus_cdf_minus, log_cdf_delta))
        return log_prob

    log_probs = (
        channel_log_prob(xr, m_r, sr, inv_scale_r)
        + channel_log_prob(xg, m_g, sg, inv_scale_g)
        + channel_log_prob(xb, m_b, sb, inv_scale_b)
    )
    log_prob = torch.logsumexp(F.log_softmax(logit_probs, dim=-1) + log_probs, dim=-1)
    return -log_prob.mean()


## 7. PixelCNN: sieć predykcyjna

Pierwsza warstwa typu A zapewnia, że sygnał pikseli wejściowych nie zawiera bieżącego celu. Kolejne bloki typu B mogą łączyć pozycję z jego ukrytym stanem, który sam jest przyczynowy. Głowica `1×1` emituje parametry mieszaniny w każdej pozycji. Dla `K` składników potrzeba `K + 3×3K = 10K` kanałów wyjściowych.

Uwaga: cała sieć zachowuje przyczynowość tylko wtedy, gdy wszystkie operacje przestrzenne są przyczynowe. Nie dodawaj zwykłej konwolucji 3×3 ani normalizacji obejmującej pozycje przestrzenne bez sprawdzenia wpływu na przyszłe piksele.


In [ ]:
class PixelCNN(nn.Module):
    def __init__(self, channels=64, num_residual=6, num_mixtures=10):
        super().__init__()
        self.num_mixtures = num_mixtures
        self.input_conv = MaskedConv2d(3, channels, kernel_size=7, padding=3, mask_type="A")
        self.residual_blocks = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_residual)])
        self.output_net = nn.Sequential(
            nn.ReLU(inplace=True),
            MaskedConv2d(channels, channels, kernel_size=1, mask_type="B"),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, 10 * num_mixtures, kernel_size=1),
        )

    def forward(self, x):
        x = x * 2.0 - 1.0  # modelowanie w [-1, 1]
        return self.output_net(self.residual_blocks(self.input_conv(x)))

model = PixelCNN(channels=64, num_residual=6, num_mixtures=10).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametry: {num_params:,}")
with torch.no_grad():
    dummy = torch.zeros(2, 3, 32, 32, device=device)
    out = model(dummy)
print("output:", tuple(out.shape), "| oczekiwane kanały:", 10 * model.num_mixtures)


## 8. Trening i metryki

Trening jest w pełni równoległy po pozycjach: dla każdego obrazu obliczamy log-prawdopodobieństwo wszystkich pikseli w jednym przebiegu. `bits/dim` pozwala porównywać jakość gęstości niezależnie od liczby kanałów i rozmiaru obrazu. Mniejsza wartość oznacza lepsze modelowanie danych.

Poniższa konfiguracja ma być punktem startowym, nie gwarancją najlepszego wyniku. Jakość zależy od czasu treningu, architektury i sprzętu. Najpierw uruchom kilka epok, sprawdź działanie całego przepływu, a potem zwiększ budżet.


In [ ]:
EPOCHS = 10
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 1.0
CHECKPOINT_PATH = Path("pixelcnn_cifar10.pt")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def bits_per_dim(nll_nats):
    # Funkcja straty uśrednia po pozycjach (H,W), ale sumuje RGB.
    return nll_nats / (3 * math.log(2.0))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_nll, total_examples = 0.0, 0
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        params = model(x)
        nll = discretized_mix_logistic_loss(params, x * 2 - 1, model.num_mixtures)
        total_nll += nll.item() * x.size(0)
        total_examples += x.size(0)
    return total_nll / total_examples

def train_one_epoch(model, loader):
    model.train()
    total_nll, total_examples = 0.0, 0
    for step, (x, _) in enumerate(loader, start=1):
        x = x.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        params = model(x)
        loss = discretized_mix_logistic_loss(params, x * 2 - 1, model.num_mixtures)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        total_nll += loss.item() * x.size(0)
        total_examples += x.size(0)
        if step % 100 == 0:
            print(f"  batch {step:4d}/{len(loader)} | nll={loss.item():.3f} nat/pixel")
    return total_nll / total_examples


In [ ]:
history = {"train_bpd": [], "val_bpd": []}
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_nll = train_one_epoch(model, train_loader)
    val_nll = evaluate(model, val_loader)
    history["train_bpd"].append(bits_per_dim(train_nll))
    history["val_bpd"].append(bits_per_dim(val_nll))
    elapsed = time.time() - start
    print(f"Epoch {epoch:02d}/{EPOCHS} | train {history['train_bpd'][-1]:.4f} bpd | "
          f"val {history['val_bpd'][-1]:.4f} bpd | {elapsed/60:.1f} min")
    if val_nll < best_val:
        best_val = val_nll
        torch.save({"model": model.state_dict(), "config": {
            "channels": 64, "num_residual": 6, "num_mixtures": model.num_mixtures,
            "epoch": epoch, "val_bpd": bits_per_dim(val_nll),
        }}, CHECKPOINT_PATH)
        print(f"  zapisano najlepszy checkpoint: {CHECKPOINT_PATH}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_bpd"], marker="o", label="train")
plt.plot(history["val_bpd"], marker="o", label="validation")
plt.xlabel("Epoka")
plt.ylabel("Bits per dimension (mniej = lepiej)")
plt.title("PixelCNN na CIFAR-10")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


### Jak czytać krzywe?

- Malejące train i validation bpd wskazują, że model uczy się struktury obrazów.
- Spadający train i rosnący validation bpd sugerują przeuczenie; rozważ mniejszy model, augmentację (z zachowaniem poprawnej interpretacji gęstości) lub regularyzację.
- Wysokie bpd po krótkim treningu nie oznacza błędu: CIFAR-10 jest zróżnicowany, a PixelCNN wymaga wielu aktualizacji.
- Do porównania publikowanych wyników należy sprawdzić dokładny wariant danych, likelihood, preprocesing i sposób raportowania. Sama liczba bez protokołu bywa myląca.


## 9. Próbkowanie: od rozkładu do obrazu

Próbkowanie jest sekwencyjne. Dla każdego piksela liczymy predykcję, losujemy indeks składnika mieszaniny, a następnie losujemy kanały RGB z logistycznych rozkładów. W tym prostym kodzie ponownie wykonujemy forward całej sieci dla każdego piksela, co jest bardzo wolne. To czytelna implementacja referencyjna; wydajne PixelCNN buforują aktywacje lub używają zoptymalizowanych jąder.

Wagi mieszaniny pobieramy przez softmax. Losowanie logistyczne wykonujemy z równomiernej zmiennej przez odwrotną dystrybuantę. Wartości są obcinane do zakresu 8-bitowego, a na końcu kwantyzowane.


In [ ]:
@torch.no_grad()
def sample_logistic(mean, log_scale):
    u = torch.rand_like(mean).clamp_(1e-5, 1 - 1e-5)
    return mean + torch.exp(log_scale) * (torch.log(u) - torch.log1p(-u))

@torch.no_grad()
def sample_pixelcnn(model, n_samples=4, height=32, width=32, temperature=1.0):
    model.eval()
    x = torch.zeros(n_samples, 3, height, width, device=device)
    k = model.num_mixtures
    for row in range(height):
        for col in range(width):
            params = model(x)
            p = params[:, :, row, col]
            logits = p[:, :k] / temperature
            values = p[:, k:].view(n_samples, 3, 3 * k)
            means, log_scales, coeffs = values[:, :, :k], values[:, :, k:2*k], torch.tanh(values[:, :, 2*k:])
            log_scales = log_scales.clamp(min=-7.0)
            mixture = torch.distributions.Categorical(logits=logits).sample()
            gather_index = mixture[:, None, None].expand(-1, 3, 1)
            means = means.gather(-1, gather_index).squeeze(-1)
            log_scales = log_scales.gather(-1, gather_index).squeeze(-1)
            coeffs = coeffs.gather(-1, gather_index).squeeze(-1)

            r = sample_logistic(means[:, 0], log_scales[:, 0]).clamp(-1, 1)
            g_mean = means[:, 1] + coeffs[:, 0] * r
            g = sample_logistic(g_mean, log_scales[:, 1]).clamp(-1, 1)
            b_mean = means[:, 2] + coeffs[:, 1] * r + coeffs[:, 2] * g
            blue = sample_logistic(b_mean, log_scales[:, 2]).clamp(-1, 1)
            pixel = torch.stack([r, g, blue], dim=1)
            pixel = torch.round((pixel + 1) * 127.5).clamp(0, 255) / 255.0
            x[:, :, row, col] = pixel
    return x


In [ ]:
# Wczytanie najlepszego checkpointu, jeśli trening został wcześniej wykonany.
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model"])
    print("Wczytano epoch", checkpoint["config"]["epoch"],
          "| val bpd", round(checkpoint["config"]["val_bpd"], 4))
else:
    print("Brak checkpointu — próbki będą pochodziły z aktualnych wag.")

samples = sample_pixelcnn(model, n_samples=4)
grid = make_grid(samples.cpu(), nrow=2, padding=2)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Próbki PixelCNN")
plt.show()


## 10. Analiza modelu i jego błędów

### 10.1 Log-wiarygodność zbioru testowego

Test używamy dopiero po zakończeniu decyzji projektowych. Raportujemy bpd jako średnią po obrazach.


In [ ]:
test_nll = evaluate(model, test_loader)
print(f"CIFAR-10 test: {bits_per_dim(test_nll):.4f} bits/dim")


### 10.2 Porównanie rekonstrukcji kontekstu

PixelCNN nie jest autoenkoderem i nie rekonstruuje obrazu wprost. Poniższa wizualizacja pokazuje obrazy rzeczywiste obok wygenerowanych próbek. Zwróć uwagę na lokalne tekstury, spójność kształtów oraz powtarzające się artefakty. Niska NLL nie gwarantuje wizualnie najlepszych próbek: likelihood i ocena percepcyjna mierzą różne aspekty.


In [ ]:
real, _ = next(iter(test_loader))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0, i].imshow(real[i].permute(1, 2, 0)); axes[0, i].set_title("CIFAR-10"); axes[0, i].axis("off")
    axes[1, i].imshow(samples[i].cpu().permute(1, 2, 0)); axes[1, i].set_title("PixelCNN"); axes[1, i].axis("off")
plt.tight_layout()


### 10.3 Diagnostyka generowania

Przyjrzyj się osobno: (1) czy próbki mają realistyczne lokalne tekstury, (2) czy obiekty zachowują spójny kształt, (3) czy występują wzory szachownicy lub pasy, (4) czy różne próbki są zróżnicowane. Słaba globalna struktura przy wiarygodnych małych fragmentach jest częstym objawem zbyt małego pola recepcyjnego albo niewystarczającego treningu.

Można też porównać temperaturę próbkowania. Wartość mniejsza od 1 zwykle daje bardziej zachowawcze, ale mniej różnorodne próbki. Temperatura wpływa na generowanie, nie na wyliczoną NLL.


## 11. Co warto zapamiętać

1. **Autoregresja** daje dokładną faktoryzację rozkładu, ale narzuca kolejność generowania.
2. **Maska A** chroni predykcję przed bieżącym pikselem; **maska B** pozwala ukrytym cechom korzystać z bieżącej pozycji, bo są już przyczynowe.
3. **Residual blocks** pozwalają budować głębsze sieci i poprawiają przepływ gradientu.
4. **Mixture of logistics** modeluje dyskretną intensywność i zależność kanałów efektywniej niż niezależny softmax 256 klas dla każdego RGB.
5. **Trening równoległy, generowanie sekwencyjne**: to kluczowa różnica praktyczna względem wielu innych modeli generatywnych.
6. **BPD** to użyteczna miara ilościowa, ale warto oglądać próbki i sprawdzać protokół oceny.

### Ograniczenia tej implementacji

- Wolne próbkowanie przez pełny forward dla każdego piksela.
- Jedna maskowana konwolucja 2D nie rozdziela osobno pionowego i poziomego strumienia; to prostsza wersja PixelCNN.
- Krótki trening i niewielka szerokość/głębokość będą ograniczać jakość próbek.
- Brak augmentacji i warunkowania etykietą klasy.
- Obliczanie CDF różnicą może być numerycznie trudne; implementacja używa przybliżenia gęstości w wąskim przedziale i stabilnych log-sigmoidów.


## 12. Ćwiczenia rozszerzające

1. **Maska i kolejność:** zmień porządek na kolumnowy. Zaktualizuj maskę i test przyczynowości.
2. **Głębszy kontekst:** dodaj dylatowane maskowane konwolucje. Jak zmienia się pole recepcyjne i bpd?
3. **Warunkowanie klasą:** dodaj embedding etykiety CIFAR-10 do bloków rezydualnych (np. jako bias kanałowy). Porównaj próbki dla wybranych klas.
4. **Rozmiar modelu:** zestaw 3, 6 i 12 bloków rezydualnych przy podobnym budżecie treningowym. Obserwuj krzywe i koszt.
5. **Liczba mieszanin:** porównaj `K=1, 5, 10`. Jak zmienia się koszt i jakość likelihood?
6. **Szybsze wnioskowanie:** cache'uj aktywacje w kolejności rasterowej lub zaimplementuj specjalizowane jądro. Zmierz przyspieszenie względem `sample_pixelcnn`.
7. **Porównanie architektur:** zestaw podstawowy PixelCNN z gated PixelCNN albo PixelCNN++ i opisz różnice w polu recepcyjnym oraz likelihood.
8. **Analiza błędów:** pokaż próbki przy kilku temperaturach oraz przykłady o najgorszym log-likelihood ze zbioru testowego.

### Źródła do dalszej lektury

- van den Oord i in., [Conditional Image Generation with PixelCNN Decoders (2016)](https://arxiv.org/abs/1606.05328)
- Salimans i in., [PixelCNN++: Improving the PixelCNN with Discretized Logistic Mixture Likelihood and Other Modifications (2017)](https://arxiv.org/abs/1701.05517)
- van den Oord i in., [Pixel Recurrent Neural Networks (2016)](https://arxiv.org/abs/1601.06759)
